# Liber · Block 3 ViT 분류기 본학습 (Colab)

Block 2를 통과한 조각을 **`book` · `not_book` · `partial` · `multiple`** 4종류로 나눠 권수를 확정하는 ViT를 본학습한다.

| 모델 | 용도 | 최대 에폭 | 예상 시간 (A100) |
|---|---|---|---|
| **ViT-Large** (304M) | 설계(아키텍처)대로 · PC 노트북 · 이후 양자화 | 30 | 1~2시간 (조기 종료 시 더 짧음) |
| **DeiT-Small** (22M, ViT 구조) | 웹앱 β2용 (휴대폰에 올릴 크기) | 50 | 20~40분 |

- 검증 정확도가 `PATIENCE` 에폭 동안 오르지 않으면 멈추고 **가장 좋았던 에폭**을 저장 (사전학습 모델 미세조정은 보통 10~20에폭이면 수렴)
- 학습 데이터: 검출기 노트북이 만든 데이터셋(`books_obb_built.zip`, 1~3권 보강 포함)의 정답 상자에서 4종류 조각 자동 생성
  + **책이 없는 장면**(책상·모니터·건물 등 Open Images)을 `not_book`에 보강 — 1차 테스트에서 판자·모니터·표지 그림을 책으로 본 문제 대응
  + **Kaggle Book Covers**(CC0, 표지 3.2만 장 중 6,000장)로 표지 `book` · 표지 일부 `partial` · 표지 2~3장 나란히 `multiple` 보강
    — 표지 그림(건물·인물)을 책의 일부로 배우게 해 1591 같은 오판(표지 속 건물 창문 → `multiple`)을 줄임
- 입력 전처리는 맥 노트북 Cell 3-2와 **똑같다** (회색 여백 정사각형 224 · 정규화) → 맥에서 그대로 불러 씀

### 실행 전
1. **검출기 노트북(`colab_train_det.ipynb`)을 먼저 Cell 7까지 실행**해 Drive `내 드라이브/liber/books_obb_built.zip`이 있어야 함
   (검출기 학습이 도는 중이어도 됨 → 이 노트북은 **새 런타임(새 탭)**에서 동시에 실행 가능, Colab 컴퓨팅 단위는 둘 다 차감)
2. (선택) Kaggle `archive.zip`을 Drive `내 드라이브/liber/book_covers.zip`으로 이름을 바꿔 올림 (없으면 표지 보강만 건너뜀)
3. 런타임 유형 **A100 GPU** (L4도 가능, ViT-Large가 2~3배 느림) → **모두 실행**

### 결과 (Drive `내 드라이브/liber/`)
`liber_vit_results.zip` = `chunk_vit_large/` · `chunk_vit_small/` (맥 `weights/chunk_vit`에 넣을 폴더) · 혼동 행렬 · 분류별 정확도 · 단계별 시간

In [ ]:
# Cell 1 · 환경 준비: GPU · Google Drive · 경로
!nvidia-smi --query-gpu=name,memory.total --format=csv

import json
import math
import os
import random
import shutil
import subprocess
import time
from concurrent.futures import ThreadPoolExecutor
from multiprocessing import Pool
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import torch
from google.colab import drive
from PIL import Image
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from transformers import AutoImageProcessor, AutoModelForImageClassification

drive.mount("/content/drive")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
TIMES = {}

DRIVE = Path("/content/drive/MyDrive/liber")
WORK = Path("/content/work_vit")
DS = WORK / "books_obb"            # 검출기 노트북이 만든 데이터셋을 여기에 복원
CHUNK = WORK / "chunk_cls"         # 4종류 조각 이미지
BUILT_ZIP = DRIVE / "books_obb_built.zip"
WORK.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("GPU:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "없음 → 런타임 유형을 A100 GPU로")
print(f"Drive 여유 공간 약 {shutil.disk_usage('/content/drive').free / 1e9:.1f}GB (이 노트북이 쓰는 양: 약 1.5GB)")

In [ ]:
# Cell 2 · 설정
MODELS = [   # (이름, 체크포인트, 최대 에폭, 학습률, 배치)
    ("large", "google/vit-large-patch16-224-in21k", 30, 3e-5, 64),
    ("small", "facebook/deit-small-patch16-224", 50, 1e-4, 256),
]
PATIENCE = 4            # 검증 정확도가 이만큼 에폭 동안 안 오르면 멈춤
VIT_SIZE = 224
CHUNK_CLASSES = ["book", "not_book", "partial", "multiple"]
CHUNK_MAX_IMAGES = {"train": 12000, "val": 1500}   # 조각을 뽑을 원본 사진 수 (사진 1장 → 최대 8조각)
PER_IMAGE = {"book": 3, "not_book": 2, "partial": 2, "multiple": 1}
CROP_MAX_SIDE = 320
NEG_IMAGES, NEG_PER_IMAGE, NEG_VAL_RATIO = 2500, 4, 0.15   # 책 없는 장면 → not_book
COVERS_ZIP = DRIVE / "book_covers.zip"                       # Kaggle Book Covers (CC0) — 학습에만 쓰고 배포하지 않음
COVER_N, COVER_VAL_RATIO = 6000, 0.1
QUICK_CHECK = False     # True: 데이터 5% · 모델별 1에폭으로 끝까지 도는지만 확인

if QUICK_CHECK:
    MODELS = [(n, c, 1, lr, b) for n, c, _, lr, b in MODELS]
    CHUNK_MAX_IMAGES = {"train": 600, "val": 150}
    NEG_IMAGES = 100
    COVER_N = 300

assert BUILT_ZIP.exists(), f"{BUILT_ZIP}이 없습니다 → 검출기 노트북(colab_train_det.ipynb)을 Cell 7까지 먼저 실행하세요"
if not DS.exists():
    t0 = time.time()
    subprocess.run(["unzip", "-q", "-o", str(BUILT_ZIP), "-d", str(WORK)], check=True)
    TIMES["데이터셋 복원"] = (time.time() - t0) / 60
print({s: len(list((DS / "images" / s).iterdir())) for s in ("train", "val")})

In [ ]:
# Cell 3 · 조각 만들기 함수 (맥 노트북 Cell 3-1과 같은 규칙)
def rect_frame(poly):
    """회전 상자 → (p0, u, v): u = 긴 변, v = 짧은 변 (좌우 반전 없는 방향)"""
    box = cv2.boxPoints(cv2.minAreaRect(np.asarray(poly, np.float32)))
    for i in range(4):
        for step in (1, -1):
            p0, p1, p3 = box[i], box[(i + step) % 4], box[(i - step) % 4]
            u, v = p1 - p0, p3 - p0
            if np.linalg.norm(u) >= np.linalg.norm(v) and v[0] * u[1] - v[1] * u[0] > 0:
                return p0, u, v
    p0, p1, p3 = box[0], box[1], box[3]
    return p0, p1 - p0, p3 - p0


def frame_poly(p0, u, v):
    return np.array([p0, p0 + u, p0 + u + v, p0 + v], np.float32)


def crop_rotated(img, poly):
    p0, u, v = rect_frame(poly)
    L, S = max(int(round(np.linalg.norm(u))), 1), max(int(round(np.linalg.norm(v))), 1)
    M = cv2.getAffineTransform(np.float32([p0, p0 + u, p0 + v]), np.float32([[0, 0], [0, L], [S, 0]]))
    return cv2.warpAffine(img, M, (S, L), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)


def limit_side(img, max_side):
    s = max_side / max(img.shape[:2])
    return cv2.resize(img, None, fx=s, fy=s, interpolation=cv2.INTER_AREA) if s < 1 else img


def read_obb(lbl_path, w, h):
    return [(np.array(l.split()[1:9], np.float32).reshape(4, 2) * np.float32([w, h])).astype(np.float32)
            for l in Path(lbl_path).read_text().splitlines() if len(l.split()) >= 9]


def make_chunk_samples(img, polys, rng_):
    h, w = img.shape[:2]
    out = {k: [] for k in CHUNK_CLASSES}
    for i in rng_.sample(range(len(polys)), min(PER_IMAGE["book"], len(polys))):
        out["book"].append(polys[i])
    for i in rng_.sample(range(len(polys)), min(PER_IMAGE["partial"], len(polys))):
        p0, u, v = rect_frame(polys[i])
        frac = rng_.uniform(0.25, 0.6)
        out["partial"].append(frame_poly(p0 + u * rng_.uniform(0, 1 - frac), u * frac, v))
    order = sorted(range(len(polys)), key=lambda i: polys[i][:, 0].mean())
    for _ in range(5):
        if len(out["multiple"]) >= PER_IMAGE["multiple"] or len(order) < 2:
            break
        k = rng_.randint(2, min(4, len(order)))
        start = rng_.randint(0, len(order) - k)
        group = [polys[i] for i in order[start:start + k]]
        rect = cv2.minAreaRect(np.concatenate(group))
        if rect[1][0] * rect[1][1] <= 1.5 * sum(cv2.contourArea(g) for g in group):
            out["multiple"].append(cv2.boxPoints(rect))
    s = 0.25
    gt = np.zeros((max(int(h * s), 1), max(int(w * s), 1)), np.uint8)
    cv2.fillPoly(gt, [(p * s).astype(np.int32) for p in polys], 1)
    for _ in range(20):
        if len(out["not_book"]) >= PER_IMAGE["not_book"]:
            break
        (_, _), (bw, bh), ang = cv2.minAreaRect(polys[rng_.randrange(len(polys))])
        k = rng_.uniform(0.5, 1.5)
        box = cv2.boxPoints(((rng_.uniform(0, w), rng_.uniform(0, h)), (bw * k, bh * k), ang + rng_.uniform(-20, 20)))
        if box[:, 0].min() < 0 or box[:, 1].min() < 0 or box[:, 0].max() > w or box[:, 1].max() > h:
            continue
        m = np.zeros_like(gt)
        cv2.fillPoly(m, [(box * s).astype(np.int32)], 1)
        if m.sum() and (gt & m).sum() / m.sum() < 0.1:
            out["not_book"].append(box)
    return out


def chunk_one(args):
    """사진 1장 → 4종류 조각 저장 (병렬 실행용)"""
    split, img_path, seed = args
    img = cv2.imread(str(img_path))
    if img is None:
        return 0
    polys = read_obb(DS / "labels" / split / f"{img_path.stem}.txt", img.shape[1], img.shape[0])
    if not polys:
        return 0
    n = 0
    for k, boxes in make_chunk_samples(img, polys, random.Random(seed)).items():
        for j, box in enumerate(boxes):
            crop = crop_rotated(img, box)
            if min(crop.shape[:2]) >= 8:
                cv2.imwrite(str(CHUNK / split / k / f"{img_path.stem}_{j}.jpg"), limit_side(crop, CROP_MAX_SIDE),
                            [cv2.IMWRITE_JPEG_QUALITY, 90])
                n += 1
    return n

In [ ]:
# Cell 4 · [데이터 ①] 검출기 데이터셋의 정답 상자 → 4종류 조각 (반복 복사본 _rep 제외, 병렬)
t0 = time.time()
if CHUNK.exists():
    shutil.rmtree(CHUNK)
for split in ("train", "val"):
    for k in CHUNK_CLASSES:
        (CHUNK / split / k).mkdir(parents=True)
rng = random.Random(SEED)
jobs = []
for split in ("train", "val"):
    imgs = sorted(p for p in (DS / "images" / split).iterdir() if "_rep" not in p.stem)
    rng.shuffle(imgs)
    jobs += [(split, p, SEED + i) for i, p in enumerate(imgs[:CHUNK_MAX_IMAGES[split]])]
NCPU = os.cpu_count() or 4
with Pool(NCPU) as pool:
    total = sum(pool.imap_unordered(chunk_one, jobs, chunksize=32))
TIMES["① 정답 상자 → 조각"] = (time.time() - t0) / 60
print(f"조각 {total:,}개 · {TIMES['① 정답 상자 → 조각']:.1f}분 (CPU {NCPU}개)")

In [ ]:
# Cell 5 · [데이터 ②] 책이 없는 장면(Open Images) → not_book 보강
OI_BASE = "https://storage.googleapis.com/openimages"
NEG_CLASSES = ["Table", "Desk", "Kitchen & dining room table", "Countertop", "Computer monitor", "Laptop",
               "Television", "Box", "Cabinetry", "Chest of drawers", "Whiteboard", "Picture frame", "Poster",
               "Skyscraper", "Office building", "Building", "Window"]
EXCLUDE_CLASSES = ["Book", "Bookcase", "Shelf"]
t0 = time.time()
csv_dir = WORK / "oi_csv"
csv_dir.mkdir(exist_ok=True)
files = {"classes": f"{OI_BASE}/v7/oidv7-class-descriptions-boxable.csv",
         "validation": f"{OI_BASE}/v5/validation-annotations-bbox.csv",
         "test": f"{OI_BASE}/v5/test-annotations-bbox.csv"}
for url in files.values():
    if not (csv_dir / Path(url).name).exists():
        subprocess.run(["wget", "-q", "-O", str(csv_dir / Path(url).name), url], check=True)
names = pd.read_csv(csv_dir / Path(files["classes"]).name)
mid = dict(zip(names.DisplayName, names.LabelName))
ann = pd.concat([pd.read_csv(csv_dir / Path(files[s]).name, usecols=["ImageID", "LabelName"]).assign(split=s)
                 for s in ("validation", "test")])
bad = set(ann.ImageID[ann.LabelName.isin([mid[n] for n in EXCLUDE_CLASSES])])
good = ann[ann.LabelName.isin([mid[n] for n in NEG_CLASSES]) & ~ann.ImageID.isin(bad)]
picks = good.drop_duplicates("ImageID").sample(n=min(NEG_IMAGES, good.ImageID.nunique()), random_state=SEED)
neg_dir = WORK / "oi_neg"
neg_dir.mkdir(exist_ok=True)


def fetch_neg(row):
    out = neg_dir / f"{row.ImageID}.jpg"
    if not out.exists():
        try:
            r = requests.get(f"https://open-images-dataset.s3.amazonaws.com/{row.split}/{row.ImageID}.jpg", timeout=30)
            r.raise_for_status()
        except requests.RequestException:
            return None
        img = cv2.imdecode(np.frombuffer(r.content, np.uint8), cv2.IMREAD_COLOR)
        if img is None:
            return None
        cv2.imwrite(str(out), limit_side(img, 1024), [cv2.IMWRITE_JPEG_QUALITY, 92])
    return out


with ThreadPoolExecutor(32) as ex:
    paths = [p for p in ex.map(fetch_neg, picks.itertuples(index=False)) if p is not None]
rng_n = random.Random(SEED)
n_saved = {"train": 0, "val": 0}
for i, path in enumerate(paths):
    img = cv2.imread(str(path))
    h, w = img.shape[:2]
    split = "val" if i < int(len(paths) * NEG_VAL_RATIO) else "train"
    for j in range(NEG_PER_IMAGE):
        aspect = rng_n.choice([rng_n.uniform(1.2, 1.7), rng_n.uniform(3, 12)])   # 표지 비율 / 책등 비율
        long_ = rng_n.uniform(0.15, 0.6) * max(h, w)
        box = cv2.boxPoints(((rng_n.uniform(0, w), rng_n.uniform(0, h)), (long_ / aspect, long_), rng_n.uniform(-30, 30)))
        box[:, 0] = np.clip(box[:, 0], 0, w - 1)
        box[:, 1] = np.clip(box[:, 1], 0, h - 1)
        crop = crop_rotated(img, box)
        if min(crop.shape[:2]) >= 8:
            cv2.imwrite(str(CHUNK / split / "not_book" / f"oi_neg_{path.stem}_{j}.jpg"), limit_side(crop, CROP_MAX_SIDE),
                        [cv2.IMWRITE_JPEG_QUALITY, 90])
            n_saved[split] += 1
TIMES["② not_book 보강"] = (time.time() - t0) / 60
print(f"not_book 보강 train {n_saved['train']:,} · val {n_saved['val']:,}")

In [ ]:
# Cell 5b · [데이터 ③] Kaggle Book Covers → 표지 book · 표지 일부 partial · 표지 2~3장 나란히 multiple
#   표지만 잘린 깨끗한 디지털 이미지라, 휴대폰 사진처럼 해상도를 낮췄다 키우고 가끔 흐리게 해서 넣음
t0 = time.time()
if not COVERS_ZIP.exists():
    print(f"건너뜀: {COVERS_ZIP} 없음 (Kaggle archive.zip을 이 이름으로 Drive에 올리면 사용)")
else:
    cdir = WORK / "covers"
    if not cdir.exists():
        subprocess.run(["unzip", "-q", "-o", str(COVERS_ZIP), "book-covers/*", "-d", str(cdir)], check=True)
    paths = sorted(cdir.rglob("*.jpg"))
    random.Random(SEED).shuffle(paths)
    paths = paths[:COVER_N]

    def camera(img, r):
        s = r.uniform(0.5, 1.0)
        h, w = img.shape[:2]
        img = cv2.resize(cv2.resize(img, (max(8, int(w * s)), max(8, int(h * s))), interpolation=cv2.INTER_AREA), (w, h))
        return cv2.GaussianBlur(img, (3, 3), 0) if r.random() < 0.3 else img

    n_cov = {"book": 0, "partial": 0, "multiple": 0}
    for i, p in enumerate(paths):
        img = cv2.imread(str(p))
        if img is None or min(img.shape[:2]) < 32:
            continue
        r = random.Random(SEED + i)
        split = "val" if i < int(len(paths) * COVER_VAL_RATIO) else "train"
        h = img.shape[0]
        cv2.imwrite(str(CHUNK / split / "book" / f"cover_{i}.jpg"), camera(img, r), [cv2.IMWRITE_JPEG_QUALITY, 90])
        frac = r.uniform(0.25, 0.6)                       # 책의 긴 변 방향(세로) 25~60% 구간 = partial
        y0 = int(r.uniform(0, 1 - frac) * h)
        cv2.imwrite(str(CHUNK / split / "partial" / f"cover_{i}.jpg"), camera(img[y0:y0 + max(8, int(frac * h))], r), [cv2.IMWRITE_JPEG_QUALITY, 90])
        n_cov["book"] += 1
        n_cov["partial"] += 1
        if i % 3 == 0:                                    # 표지 2~3장을 나란히 → multiple
            row = [img] + [x for x in (cv2.imread(str(paths[r.randrange(len(paths))])) for _ in range(r.randint(1, 2))) if x is not None]
            if len(row) >= 2:
                H = min(x.shape[0] for x in row)
                row = [cv2.resize(x, (max(8, int(x.shape[1] * H / x.shape[0])), H)) for x in row]
                cv2.imwrite(str(CHUNK / split / "multiple" / f"cover_{i}.jpg"), camera(cv2.hconcat(row), r), [cv2.IMWRITE_JPEG_QUALITY, 90])
                n_cov["multiple"] += 1
    print("표지 보강:", n_cov)
TIMES["③ 표지 보강"] = (time.time() - t0) / 60

In [ ]:
# Cell 6 · ✅ 데이터 확인: 분류별 장수 · 예시
counts = pd.DataFrame({s: {k: len(list((CHUNK / s / k).glob("*.jpg"))) for k in CHUNK_CLASSES} for s in ("train", "val")})
display(counts)
fig, axes = plt.subplots(4, 8, figsize=(16, 9))
for r, k in enumerate(CHUNK_CLASSES):
    ps = sorted((CHUNK / "train" / k).glob("*.jpg"))
    random.Random(r).shuffle(ps)
    for c, p in enumerate(ps[:8]):
        axes[r, c].imshow(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB))
        axes[r, c].set_title(k, fontsize=8)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 7 · [본학습] ViT-Large → DeiT-Small 차례로. 에폭마다 검증해 최고 정확도 가중치만 Drive에 저장, PATIENCE 동안 안 오르면 멈춤
#   입력 전처리 = 맥 노트북 Cell 3-2와 같음 (Letterbox 224 · ColorJitter · 좌우 반전 · 체크포인트 정규화)
class Letterbox:
    def __init__(self, size, fill=(114, 114, 114)):
        self.size, self.fill = size, fill

    def __call__(self, im):
        s = self.size / max(im.size)
        im = im.resize((max(1, round(im.width * s)), max(1, round(im.height * s))), Image.BILINEAR)
        canvas = Image.new("RGB", (self.size, self.size), self.fill)
        canvas.paste(im, ((self.size - im.width) // 2, (self.size - im.height) // 2))
        return canvas


def vit_transform(processor, train=False):
    aug = [transforms.ColorJitter(0.3, 0.3, 0.3), transforms.RandomHorizontalFlip()] if train else []
    return transforms.Compose([Letterbox(VIT_SIZE), *aug, transforms.ToTensor(),
                               transforms.Normalize(processor.image_mean, processor.image_std)])


@torch.no_grad()
def evaluate(model, dl):
    model.eval()
    preds, golds = [], []
    for x, y in dl:
        with torch.autocast("cuda", dtype=torch.bfloat16):
            preds += model(pixel_values=x.to(DEVICE, non_blocking=True)).logits.argmax(-1).cpu().tolist()
        golds += y.tolist()
    return np.array(preds), np.array(golds)


def train_vit(tag, ckpt, max_epochs, lr, batch):
    t0 = time.time()
    out_dir = WORK / f"chunk_vit_{tag}{'_quick' if QUICK_CHECK else ''}"   # 최고 가중치는 로컬에 먼저 (ViT-Large 약 1.2GB → Drive엔 끝나고 한 번만)
    processor = AutoImageProcessor.from_pretrained(ckpt)
    train_ds = datasets.ImageFolder(CHUNK / "train", transform=vit_transform(processor, train=True))
    val_ds = datasets.ImageFolder(CHUNK / "val", transform=vit_transform(processor))
    train_dl = DataLoader(train_ds, batch_size=batch, shuffle=True, num_workers=8, pin_memory=True, drop_last=True)
    val_dl = DataLoader(val_ds, batch_size=batch, num_workers=8, pin_memory=True)
    model = AutoModelForImageClassification.from_pretrained(
        ckpt, id2label=dict(enumerate(train_ds.classes)), label2id={c: i for i, c in enumerate(train_ds.classes)},
        ignore_mismatched_sizes=True).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    total = max_epochs * len(train_dl)
    warm = max(1, int(0.05 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / total))))
    best, bad_epochs, hist = -1.0, 0, []
    for epoch in range(max_epochs):
        model.train()
        te = time.time()
        for step, (x, y) in enumerate(train_dl):
            with torch.autocast("cuda", dtype=torch.bfloat16):
                loss = model(pixel_values=x.to(DEVICE, non_blocking=True), labels=y.to(DEVICE, non_blocking=True)).loss
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()
        preds, golds = evaluate(model, val_dl)
        acc = float((preds == golds).mean())
        hist.append({"epoch": epoch + 1, "loss": round(loss.item(), 4), "val_acc": round(acc, 4), "분": round((time.time() - te) / 60, 1)})
        print(f"[{tag}] epoch {epoch + 1}/{max_epochs} · loss {loss.item():.3f} · val 정확도 {acc:.4f} · {(time.time() - te) / 60:.1f}분")
        if acc > best:
            best, bad_epochs = acc, 0
            model.save_pretrained(out_dir)
            processor.save_pretrained(out_dir)
            (out_dir / "trained_on_colab.json").write_text(json.dumps(
                {"checkpoint": ckpt, "epoch": epoch + 1, "val_acc": acc, "classes": train_ds.classes,
                 "train_images": len(train_ds), "val_images": len(val_ds)}, ensure_ascii=False, indent=1))
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print(f"[{tag}] {PATIENCE}에폭 동안 개선 없음 → 조기 종료 (최고 {best:.4f})")
                break
    pd.DataFrame(hist).to_csv(out_dir / "history.csv", index=False)
    shutil.copytree(out_dir, DRIVE / out_dir.name, dirs_exist_ok=True)
    print(f"[{tag}] 최고 가중치 → {DRIVE / out_dir.name} (검증 정확도 {best:.4f})")
    TIMES[f"본학습 · {tag}"] = (time.time() - t0) / 60
    del model, opt
    torch.cuda.empty_cache()
    return out_dir, best


RESULTS = {}
for tag, ckpt, ep, lr, bs in MODELS:
    RESULTS[tag] = train_vit(tag, ckpt, ep, lr, bs)
print({k: round(v[1], 4) for k, v in RESULTS.items()})

In [ ]:
# Cell 8 · ✅ 평가: 모델별 혼동 행렬 · 분류별 재현율/정밀도 (검증 조각)
EVAL = {}
for tag, (out_dir, best) in RESULTS.items():
    processor = AutoImageProcessor.from_pretrained(out_dir)
    model = AutoModelForImageClassification.from_pretrained(out_dir).to(DEVICE)
    val_ds = datasets.ImageFolder(CHUNK / "val", transform=vit_transform(processor))
    preds, golds = evaluate(model, DataLoader(val_ds, batch_size=128, num_workers=8))
    names = val_ds.classes
    cm = pd.crosstab(pd.Series([names[i] for i in golds], name="정답"), pd.Series([names[i] for i in preds], name="예측"))
    per = pd.DataFrame({c: {"재현율(%)": round(((golds == i) & (preds == i)).sum() / max((golds == i).sum(), 1) * 100, 1),
                            "정밀도(%)": round(((golds == i) & (preds == i)).sum() / max((preds == i).sum(), 1) * 100, 1)}
                        for i, c in enumerate(names)}).T
    print(f"[{tag}] 검증 정확도 {(preds == golds).mean():.4f}")
    display(cm)
    display(per)
    cm.to_csv(out_dir / "confusion_matrix.csv", encoding="utf-8-sig")
    per.to_csv(out_dir / "per_class.csv", encoding="utf-8-sig")
    EVAL[tag] = float((preds == golds).mean())
    del model
    torch.cuda.empty_cache()

In [ ]:
# Cell 9 · 결과 한 번에 받기: liber_vit_results.zip (Drive 저장 + 브라우저 내려받기)
#   chunk_vit_large/ · chunk_vit_small/ = 맥 weights/chunk_vit 에 넣을 폴더 (가중치 · 전처리 설정 · 학습 기록 · 혼동 행렬)
from google.colab import files

BUNDLE = WORK / "liber_vit_results"
if BUNDLE.exists():
    shutil.rmtree(BUNDLE)
BUNDLE.mkdir()
for tag, (out_dir, best) in RESULTS.items():
    shutil.copytree(out_dir, BUNDLE / out_dir.name)
counts.to_csv(BUNDLE / "chunk_counts.csv", encoding="utf-8-sig")
times = pd.Series(TIMES, name="분").round(1)
times.to_csv(BUNDLE / "times.csv", encoding="utf-8-sig")
zip_path = shutil.make_archive(str(WORK / "liber_vit_results"), "zip", BUNDLE)
shutil.copy2(zip_path, DRIVE / "liber_vit_results.zip")
print(f"묶음 저장 → {DRIVE / 'liber_vit_results.zip'} ({Path(zip_path).stat().st_size / 1e9:.2f}GB)")
display(times.to_frame())
files.download(zip_path)   # ViT-Large가 약 1.2GB라 브라우저 다운로드가 느리면 Drive에서 받기

## 맥북으로 가져가기

```bash
cd ~/Downloads/shelf_ocr_test/liber_deprehensus
unzip -o ~/Downloads/liber_vit_results.zip -d outputs/colab_vit
rm -rf weights/chunk_vit && cp -R outputs/colab_vit/chunk_vit_large weights/chunk_vit   # 설계대로 ViT-Large 사용
# (가볍게 쓰려면 chunk_vit_large 대신 chunk_vit_small)
```

맥 노트북: 커널 재시작 → Cell 0 → … → **3-2(Colab 가중치가 있으면 학습을 건너뜀)** → 3-3 → 3-4